In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import networkx as nx
import dask.dataframe as dd
from scipy.sparse import csr_matrix
from functools import wraps
import time

# Define the timing decorator
def time_it(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start_time = time.perf_counter()
        result = func(*args, **kwargs)
        end_time = time.perf_counter()
        print(f"{func.__name__} executes in {end_time - start_time:.10f} seconds")
        return result
    return wrapper

In [2]:
class ContentBasedFiltering:
    def __init__(self, item_features):
        self.item_features = csr_matrix(item_features.fillna(0))
    
    def calculate_item_similarity(self, item_index):
        item_vector = self.item_features[item_index].toarray()
        return cosine_similarity(item_vector, self.item_features).flatten()
    
    def predict_rating(self, user_index, item_index, user_item_matrix):
        item_similarity = self.calculate_item_similarity(item_index)
        user_rated_items = user_item_matrix[user_index].toarray().flatten()
        non_zero_items = user_rated_items != 0
        similar_items = item_similarity[non_zero_items]
        user_ratings = user_rated_items[non_zero_items]
        return np.dot(similar_items, user_ratings) / np.sum(similar_items) if len(user_ratings) > 0 else 0

class CollaborativeFiltering:
    def __init__(self, user_item_matrix):
        self.user_item_matrix = csr_matrix(user_item_matrix.fillna(0))
    
    def calculate_user_similarity(self, user_index):
        user_vector = self.user_item_matrix[user_index].toarray()
        return cosine_similarity(user_vector, self.user_item_matrix).flatten()
    
    def predict_rating(self, user_index, item_index):
        user_similarity = self.calculate_user_similarity(user_index)
        user_ratings = self.user_item_matrix[:, item_index].toarray().flatten()
        non_zero_ratings = user_ratings != 0
        similar_users = user_similarity[non_zero_ratings]
        ratings = user_ratings[non_zero_ratings]
        if len(ratings) > 0 and np.sum(similar_users) != 0:
            return np.dot(similar_users, ratings) / np.sum(similar_users)
        else:
            return 0  # Default value when no meaningful prediction is possible

# Hybrid Filtering Class
@time_it
class HybridFiltering:
    def __init__(self, user_item_matrix, item_features):
        self.cbf = ContentBasedFiltering(item_features)
        self.cf = CollaborativeFiltering(user_item_matrix)
    
    def predict_rating(self, user_index, item_index, alpha=0.5):
        cbf_pred = self.cbf.predict_rating(user_index, item_index, self.cf.user_item_matrix)
        cf_pred = self.cf.predict_rating(user_index, item_index)
        return alpha * cf_pred + (1 - alpha) * cbf_pred
    
    def recommend_items(self, user_index, top_n=5, alpha=0.5):
        user_ratings = self.cf.user_item_matrix[user_index].toarray().flatten()
        unrated_items = np.where(user_ratings == 0)[0]
        predicted_ratings = [self.predict_rating(user_index, item_index, alpha) for item_index in unrated_items]
        top_items_indices = np.argsort(predicted_ratings)[::-1][:top_n]
        top_items = unrated_items[top_items_indices]
        return top_items, np.array(predicted_ratings)[top_items_indices]

In [3]:
# Cell 3: Load and Preprocess Data
# Step 1: Load the dataset with Dask for large datasets
data = dd.read_csv(r'e:\PROGRAMMING\Movie Recommendation System\ml-10M100K\movielens_100k.csv')
data['UserID'] = data['UserID'].astype('int32')
data['MovieID'] = data['MovieID'].astype('int32')
data['Rating'] = data['Rating'].astype('float32')
data['Timestamp'] = data['Timestamp'].astype('int32')

# Step 2: Convert Dask DataFrame to pandas DataFrame
data = data.compute()
print("Data loaded and converted to pandas DataFrame:")
display(data.head())

Data loaded and converted to pandas DataFrame:


,UserID,MovieID,Rating,Timestamp,Title,Genres
0,6727,5673,3.0,1111863291,Punch-Drunk Love (2002),Comedy|Drama|Romance|Thriller
1,30816,1196,5.0,1144445863,Star Wars: Episode V - The Empire Strikes Back...,Action|Adventure|Sci-Fi
2,46430,1263,4.0,945142550,"Deer Hunter, The (1978)",Drama|War
3,30272,1183,3.0,953764870,"English Patient, The (1996)",Drama|Romance|War
4,1479,1210,4.0,1050689805,Star Wars: Episode VI - Return of the Jedi (1983),Action|Adventure|Sci-Fi


In [4]:
# Cell 4: Create User-Item Matrix
# Step 3: Create the user-item matrix
user_item_matrix = data.pivot_table(index='UserID', columns='MovieID', values='Rating')
print("User-Item Matrix Shape:", user_item_matrix.shape)
display(user_item_matrix.head())

User-Item Matrix Shape: (38385, 6875)


MovieID,1,2,3,4,5,6,7,8,9,10,...,63329,63339,63676,63808,63859,63876,64508,64614,64957,64983
UserID,,,,,,,,,,,,,,,,,,,,,
6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
# Cell 5: Create Item Features
# Step 4: Create some dummy item features for content-based filtering
item_features = pd.DataFrame({
    'MovieID': user_item_matrix.columns,
    'Feature1': np.random.rand(len(user_item_matrix.columns)),
    'Feature2': np.random.rand(len(user_item_matrix.columns)),
    'Feature3': np.random.rand(len(user_item_matrix.columns)),
})
item_features.set_index('MovieID', inplace=True)
print("Item Features Shape:", item_features.shape)
display(item_features.head())

Item Features Shape: (6875, 3)


,Feature1,Feature2,Feature3
MovieID,,,
1,0.108165,0.453482,0.002447
2,0.872068,0.826260,0.749387
3,0.428267,0.281946,0.174474
4,0.806928,0.570479,0.959194
5,0.158508,0.595976,0.088817


In [6]:
# Cell 6: Initialize and Train Hybrid Filtering Model
# Step 5: Initialize hybrid filtering model
hf = HybridFiltering(user_item_matrix, item_features)

# Step 6: Train model on the user-item matrix
print("Training model...")
print("model training completed.")

HybridFiltering executes in 11.8837247000 seconds
Training model...
model training completed.


In [7]:
# Cell 7: Generate and Display Recommendations
# Step 7: Get recommendations for a specific user (e.g., user index 0)
user_index = 0
top_n = 10
alpha = 0.5  # Weight for collaborative vs content-based filtering
recommended_items, predicted_ratings = hf.recommend_items(user_index=user_index, top_n=top_n, alpha=alpha)

# Display results
print(f"\nTop {top_n} Recommended Items for User at Index {user_index}:")
print("Recommended Movie IDs:", recommended_items)
print("Predicted Ratings:", predicted_ratings)

# Optional: Create a DataFrame for nicer display
recommendations_df = pd.DataFrame({
    'MovieID': recommended_items,
    'Predicted Rating': predicted_ratings
})
display(recommendations_df)


Top 10 Recommended Items for User at Index 0:
Recommended Movie IDs: [  55 3334 5035    4 1173 2569 1643 4458 2793 4137]
Predicted Ratings: [4.81355145 4.80334141 4.7981668  4.79317647 4.7853164  4.78318085
 4.78173641 4.77917279 4.77905292 4.77508628]


,MovieID,Predicted Rating
0,55,4.813551
1,3334,4.803341
2,5035,4.798167
3,4,4.793176
4,1173,4.785316
5,2569,4.783181
6,1643,4.781736
7,4458,4.779173
8,2793,4.779053
9,4137,4.775086
